# 📘 Benchmark Log — T-1  


# SpLR code

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpLR_ND(nn.Module):
    """
    SpLR activation for (..., F) tensors (e.g., Transformer FFN hidden states).

    Formula:
        y = x + α * x * exp(-β * x^2)

    Golden init:
        alpha_raw = 0.182  -> alpha = 2*tanh(alpha_raw)   ≈ 0.36 at start
        beta_raw  = -1.35  -> beta  = 0.01 + softplus(.)  ≈ 0.24 at start

    Parameters:
        alpha_raw: per-feature, trainable, broadcasts over batch & seq
                  shape = (1, 1, F)
        beta_raw : scalar, trainable, stage-shared (one per activation block)
    """
    def __init__(self, num_features: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, 1, num_features), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)   # signed amplitude (per feature)
        beta  = 0.01 + F.softplus(self.beta_raw)   # positive width (scalar)
        return x + alpha * x * torch.exp(-beta * x * x)


## HARD Text Classification (RAW Activation Power): **SpLR vs GELU**

**Benchmark ID:** T-1  
**Domain:** Text classification  
**Dataset:** `yahoo_answers_topics` (10 classes)  
**Goal:** Stress-test SpLR on a *hard* real-world text dataset under **no-dropout** conditions and compare directly to GELU.

---

## 1) WHY this benchmark exists (Golden Protocol)

Historically, SP-family activations have been weaker on **text** than on **vision**.  
This benchmark exists to answer:

> **Does SpLR remain competitive on hard text classification when you remove “helpers” (dropout, tuning, pretrained features) and test fairly against GELU?**

This is a *raw capability* check, not a “best tuned” score chase.

---

## 2) Fairness & Scientific Controls (what was held constant)

Across **both** activations:

- Same dataset split (HF `yahoo_answers_topics`)
- Same text fields: `question_title + question_content + best_answer`
- Same tokenizer: whitespace lowercasing split (`basic_tokenize`)
- Same vocabulary build procedure:
  - `vocab_build_examples = 120000` from **train only**
  - `vocab_size = 40000`, `min_freq = 2`
  - Special tokens: `<pad> <unk> <cls>`
- Same model architecture (from scratch)
- Same optimizer: **AdamW**
- Same LR and weight decay
- Same batch size, max length, training steps
- Same gradient clipping rule
- Same eval cadence
- Same seeds: **(0, 1, 2)**

**Only variable:** the activation function inside the FFN blocks:
- Baseline: **GELU**
- Test: **SpLR**

---

## 3) Model & Training Setup (as executed)

### Architecture (Transformer Encoder Classifier)
- Token embedding + positional embedding
- Blocks: **4**
- Heads: **4**
- Hidden size (`d_model`): **256**
- FFN width (`d_ff`): **1024**
- Classifier uses `[CLS]` token representation
- **Dropout = 0.0 everywhere** (including attention dropout)

### Training Regime
- Optimizer: **AdamW**
- LR: **3e-4**
- Weight decay: **0.01**
- Batch size: **64**
- Max length: **256 tokens**
- Gradient clip: **1.0**
- Mixed precision: enabled
- Budget: **30,000 steps per run**
- Eval every: **800 steps**
- Seeds: **3** (0, 1, 2)

---

## 4) SpLR Definition (exact mechanism used)

SpLR was used as the FFN activation with:
\[
y = x + \alpha x e^{-\beta x^2}
\]

**Golden init**
- `alpha_raw = 0.182` → `alpha = 2*tanh(alpha_raw)` ≈ **0.36**
- `beta_raw = -1.35` → `beta = 0.01 + softplus(beta_raw)` ≈ **0.24**

**Parameterization**
- `alpha_raw`: per-feature trainable, shape `(1,1,d_ff)` (broadcasts over batch & seq)
- `beta_raw`: scalar trainable, stage-shared (one per activation module)

---

## 5) Results

### 5.1 Final accuracy per seed (30k steps)

| Seed | GELU final acc | SpLR final acc | (SpLR − GELU) |
|------|----------------:|---------------:|--------------:|
| 0    | 0.7031          | 0.7009         | -0.0022 |
| 1    | 0.6980          | 0.6979         | -0.0001 |
| 2    | 0.7006          | 0.6992         | -0.0014 |

### 5.2 Mean ± std over seeds

- **GELU:** **0.7006 ± 0.0021**
- **SpLR:** **0.6993 ± 0.0012**

**Absolute mean gap:**  
\[
0.7006 - 0.6993 = 0.0013 \;\; (\approx 0.13\%)
\]

### 5.3 Paired-difference check (seed-matched)

Because both methods were run on the same seeds, we can look at paired differences:

- Mean difference (SpLR − GELU): **−0.00123**
- 95% CI (paired, df=2): **[−0.00387, +0.00140]**

Interpretation: within this sample size, the gap is **not stable enough to claim** SpLR is worse or better; it is **consistent with parity**.

---

## 6) Learning Dynamics (WHAT happened during training)

### Early-phase behavior (first ~800–2400 steps)
- GELU starts stronger early (seed 0 example):
- step 800: GELU 0.4996 vs SpLR 0.4555
- SpLR closes the gap steadily over the next few thousand steps.

### Mid → late training (20k–30k steps)
Both settle into the same performance band (~0.69–0.70+), with small oscillations typical for this task and training regime.

### Gradient norms (gn)
Both runs show similar-order gradient norms (typically ~1.1–2.2 range), with no sign that SpLR causes unstable explosions or collapse in this setting.

---

## 7) Facts vs Hypotheses

### Facts supported by T-1
- SpLR is **competitive** with GELU on a hard 10-class text dataset under **no dropout**.
- Over 3 seeds at 30k steps, SpLR trails by ~**0.13%** absolute accuracy on average.
- The seed-matched difference is small and plausibly within noise at this sample size.

### Hypotheses (NOT claims)
- SpLR may have a slightly slower early optimization warmup on text.
- SpLR’s asymptotic performance appears close to GELU given enough steps.

---

## 8) Scientific Strength Assessment (Golden Protocol)

**Strength: Moderate–Strong for “parity baseline”**, because:
- multi-seed (n=3)
- hard dataset
- strict fairness (single variable: activation)
- long step-based training (30k)
- no dropout (raw power)

**Not strong enough for superiority claims**, because:
- n=3 is still small
- single architecture scale (only one model size)
- single dataset

---

## 9) Valid Publishable Statement from T-1 (careful wording)

> On Yahoo Answers Topics (10-class) using a from-scratch Transformer encoder with no dropout, SpLR achieves **near-parity** with GELU across 3 seeds at 30k steps (GELU: 0.7006±0.0021, SpLR: 0.6993±0.0012), indicating SpLR is **text-capable** under strict fairness constraints.

No “better than GELU” claim is made.

---

## 10) Next steps that preserve fairness (if we want stronger evidence)

1) Increase seeds to **n=5** (same budget per run) OR repeat n=3 on a second day to confirm reproducibility.  
2) Add a second hard dataset (still text classification), e.g.:
   - `dbpedia_14` or `amazon_reviews_multi` (harder/noisier)
3) Scale model once (e.g., `d_model=384`, `n_layer=6`) to test whether the parity holds under increased capacity.

---

## 11) Conclusion

T-1 is a successful milestone:

> **SpLR no longer “fails on text.”**  
> Under strict raw-power conditions and long training, it performs **within ~0.1–0.3%** of GELU and behaves stably.

This positions SpLR as a **general-capable activation candidate**, not a vision-only effect.


The code of the benchmark

In [ ]:
# ============================================================
# T-1 — HARD TEXT BENCHMARK (SpLR vs GELU)
# Dataset: yahoo_answers_topics (10-class)  ✅ hard
# Model: from-scratch Transformer encoder classifier
# Dropout: 0.0  ✅ RAW activation power
# Seeds: 3
# Budget: step-based (MAX_STEPS)
# Diagnostics: grad norm + alpha/beta tracking
# ============================================================

import os, time, math, random, json
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# pip install datasets
from datasets import load_dataset

# ----------------------------
# 0) CONFIG (EDIT ONLY HERE)
# ----------------------------
@dataclass
class CFG:
    benchmark_id: str = "T-1"
    dataset: str = "yahoo_answers_topics"

    # tokenizer/vocab
    max_len: int = 256
    vocab_size: int = 40000
    min_freq: int = 2
    vocab_build_examples: int = 120000

    # model
    d_model: int = 256
    n_head: int = 4
    n_layer: int = 4
    d_ff: int = 1024

    # training (RAW power)
    dropout: float = 0.0
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0

    # long budget (set for Kaggle)
    max_steps: int = 30000   # 30000 is usually Kaggle-safe for 2 acts x 3 seeds
    eval_every: int = 800    # eval cadence (balance speed vs info)

    # runs
    acts: Tuple[str, ...] = ("gelu", "SpLR")
    seeds: Tuple[int, ...] = (0, 1, 2)

    # logging / saving
    out_dir: str = "./t1_logs"

cfg = CFG()

# ----------------------------
# 1) DEVICE + SEED
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ----------------------------
# 2) TOKENIZER + VOCAB (stable)
# ----------------------------
def basic_tokenize(text: str) -> List[str]:
    return text.lower().replace("\n", " ").replace("\t", " ").split()

def make_text(ex: Dict) -> str:
    t = ex.get("question_title", "")
    c = ex.get("question_content", "")
    a = ex.get("best_answer", "")
    return (t + " " + c + " " + a).strip()

def build_vocab(texts: List[str], vocab_size: int, min_freq: int) -> Dict[str,int]:
    from collections import Counter
    c = Counter()
    for t in texts:
        c.update(basic_tokenize(t))

    specials = ["<pad>", "<unk>", "<cls>"]
    stoi = {s:i for i,s in enumerate(specials)}

    words = [w for w,f in c.items() if f >= min_freq]
    words.sort(key=lambda w: c[w], reverse=True)
    words = words[: max(0, vocab_size - len(specials))]
    for w in words:
        stoi[w] = len(stoi)
    return stoi

def encode(text: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    toks = ["<cls>"] + basic_tokenize(text)
    ids = [stoi.get(tok, stoi["<unk>"]) for tok in toks[:max_len]]
    pad_id = stoi["<pad>"]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    return ids

def collate_builder(stoi: Dict[str,int], max_len: int):
    pad_id = stoi["<pad>"]
    def collate_fn(examples: List[Dict]):
        texts = [make_text(x) for x in examples]
        ids = [encode(t, stoi, max_len) for t in texts]
        y = [x["topic"] for x in examples]  # yahoo label
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.long)
    return collate_fn, pad_id

# ----------------------------
# 3) ACTIVATIONS
# ----------------------------
class SpLR_ND(nn.Module):
    """
    SpLR for (..., F) tensors (FFN hidden states).
      y = x + α*x*exp(-β*x^2)
    Golden init:
      alpha_raw=0.182 -> alpha=2*tanh(...) ~ 0.36
      beta_raw=-1.35  -> beta=0.01+softplus(...) ~ 0.24
    alpha_raw: (1,1,F) trainable per-feature
    beta_raw : scalar trainable (stage-shared)
    """
    def __init__(self, num_features: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, 1, num_features), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)
        beta  = 0.01 + F.softplus(self.beta_raw)
        return x + alpha * x * torch.exp(-beta * x * x)

def make_activation(act_name: str, d_ff: int) -> nn.Module:
    if act_name == "gelu":
        return nn.GELU()
    if act_name == "SpLR":
        return SpLR_ND(d_ff)
    raise ValueError("Unknown activation: " + act_name)

# ----------------------------
# 4) MODEL (NO DROPOUT)
# ----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, d_ff: int, act: nn.Module):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, dropout=0.0, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.act = act
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x, key_padding_mask=None):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + a
        h = self.ln2(x)
        h = self.fc1(h)
        h = self.act(h)
        h = self.fc2(h)
        x = x + h
        return x

class TinyTextTransformer(nn.Module):
    def __init__(self, vocab_size: int, n_class: int, max_len: int,
                 d_model: int, n_head: int, n_layer: int, d_ff: int, act_name: str):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)

        act = make_activation(act_name, d_ff)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, d_ff, act) for _ in range(n_layer)])

        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_class)

        # Keep a reference for diagnostics
        self.act_name = act_name
        self._act_module = act  # only meaningful for SpLR diagnostics

    def forward(self, ids: torch.Tensor, pad_id: int):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, T)
        x = self.tok(ids) + self.pos(pos)

        key_padding_mask = (ids == pad_id)
        for blk in self.blocks:
            x = blk(x, key_padding_mask=key_padding_mask)

        x = self.ln(x)
        cls = x[:, 0, :]
        return self.head(cls)

# ----------------------------
# 5) EVAL + DIAGNOSTICS
# ----------------------------
@torch.no_grad()
def evaluate(model, loader, pad_id):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for ids, y in loader:
        ids, y = ids.to(DEVICE), y.to(DEVICE)
        logits = model(ids, pad_id)
        loss = F.cross_entropy(logits, y)
        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

def SpLR_stats(model: nn.Module) -> Dict[str, float]:
    # Find SpLR modules (if any)
    alphas = []
    betas = []
    for m in model.modules():
        if isinstance(m, SpLR_ND):
            alpha = 2.0 * torch.tanh(m.alpha_raw.detach())
            beta  = 0.01 + F.softplus(m.beta_raw.detach())
            alphas.append(alpha)
            betas.append(beta)

    if not alphas:
        return {}

    a = torch.cat([x.reshape(-1) for x in alphas], dim=0)
    b = torch.stack([x.reshape(()) for x in betas])

    return {
        "alpha_mean": float(a.mean().cpu()),
        "alpha_std":  float(a.std(unbiased=False).cpu()),
        "alpha_min":  float(a.min().cpu()),
        "alpha_max":  float(a.max().cpu()),
        "beta_mean":  float(b.mean().cpu()),
        "beta_min":   float(b.min().cpu()),
        "beta_max":   float(b.max().cpu()),
    }

# ----------------------------
# 6) TRAIN ONE RUN (LONG, STEP-BASED)
# ----------------------------
def train_one_run(act_name: str, seed: int) -> Dict[str, Any]:
    set_seed(seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    ds = load_dataset(cfg.dataset)
    n_class = ds["train"].features["topic"].num_classes

    # Vocab built from train slice (train only)
    n_vocab = min(cfg.vocab_build_examples, len(ds["train"]))
    train_slice = ds["train"].select(range(n_vocab))
    texts_for_vocab = [make_text(ex) for ex in train_slice]
    stoi = build_vocab(texts_for_vocab, cfg.vocab_size, cfg.min_freq)

    collate_fn, pad_id = collate_builder(stoi, cfg.max_len)

    train_loader = torch.utils.data.DataLoader(
        ds["train"], batch_size=cfg.batch_size, shuffle=True,
        num_workers=2, pin_memory=(DEVICE=="cuda"), collate_fn=collate_fn
    )
    test_loader = torch.utils.data.DataLoader(
        ds["test"], batch_size=cfg.batch_size, shuffle=False,
        num_workers=2, pin_memory=(DEVICE=="cuda"), collate_fn=collate_fn
    )

    model = TinyTextTransformer(
        vocab_size=len(stoi), n_class=n_class, max_len=cfg.max_len,
        d_model=cfg.d_model, n_head=cfg.n_head, n_layer=cfg.n_layer, d_ff=cfg.d_ff,
        act_name=act_name
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

    history = []
    best_acc = 0.0
    best_snapshot = None

    step = 0
    t0 = time.time()

    model.train()
    while step < cfg.max_steps:
        for ids, y in train_loader:
            step += 1
            if step > cfg.max_steps:
                break

            ids, y = ids.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                logits = model(ids, pad_id)
                loss = F.cross_entropy(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)

            # grad norm (diagnostic)
            grad_norm = float(nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).detach().cpu())

            scaler.step(opt)
            scaler.update()

            if step % cfg.eval_every == 0 or step == cfg.max_steps:
                test_loss, test_acc = evaluate(model, test_loader, pad_id)
                snap = {
                    "step": step,
                    "train_loss": float(loss.detach().cpu()),
                    "test_loss": float(test_loss),
                    "test_acc": float(test_acc),
                    "grad_norm": grad_norm,
                }

                # SpLR diagnostics
                if act_name == "SpLR":
                    snap.update(SpLR_stats(model))

                history.append(snap)
                best_acc = max(best_acc, test_acc)

                print(
                    f"[{cfg.benchmark_id}] {act_name} seed={seed} "
                    f"step={step}/{cfg.max_steps} "
                    f"train_loss={snap['train_loss']:.4f} "
                    f"test_acc={snap['test_acc']:.4f} "
                    f"best={best_acc:.4f} "
                    f"gn={snap['grad_norm']:.3f}"
                )

    # Final metrics
    test_loss, test_acc = evaluate(model, test_loader, pad_id)
    dt = time.time() - t0
    params = sum(p.numel() for p in model.parameters())

    result = {
        "benchmark_id": cfg.benchmark_id,
        "act": act_name,
        "seed": seed,
        "dataset": cfg.dataset,
        "max_len": cfg.max_len,
        "vocab_size": cfg.vocab_size,
        "dropout": cfg.dropout,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "max_steps": cfg.max_steps,
        "eval_every": cfg.eval_every,
        "d_model": cfg.d_model,
        "n_head": cfg.n_head,
        "n_layer": cfg.n_layer,
        "d_ff": cfg.d_ff,
        "params": int(params),
        "time_sec": float(dt),
        "final_test_acc": float(test_acc),
        "final_test_loss": float(test_loss),
        "history": history,
    }

    # Save JSON for later analysis
    out_path = os.path.join(cfg.out_dir, f"{cfg.benchmark_id}_{act_name}_seed{seed}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Saved: {out_path}")
    return result

# ----------------------------
# 7) RUN ALL + SUMMARY (mean ± std)
# ----------------------------
def run_all():
    all_results = []
    for act in cfg.acts:
        for seed in cfg.seeds:
            print("\n" + "="*80)
            print(f"RUN: {cfg.benchmark_id} | act={act} | seed={seed} | steps={cfg.max_steps} | dropout=0.0")
            print("="*80)
            r = train_one_run(act, seed)
            all_results.append(r)

    import statistics as stats
    print("\n" + "="*80)
    print(f"SUMMARY — {cfg.benchmark_id} (final_test_acc mean ± std over seeds)")
    print("="*80)
    for act in cfg.acts:
        xs = [r["final_test_acc"] for r in all_results if r["act"] == act]
        print(f"{act:>4}: {stats.mean(xs):.4f} ± {stats.pstdev(xs):.4f}  (n={len(xs)})")

    return all_results

results = run_all()


# CONCLUSION 

More stable on text and images so far!